# ISLES 2022 预处理（Colab）

与 `colab_amos_preprocess.ipynb`、`colab_ribfrac_preprocess.ipynb`、`preprocess/proc_spatial_sequence.py` **相同的处理逻辑与超参**：
- 每个 nii volume：**50 个 npy**（z≥128 时）或 **33 个 npy**（z<128 时，`int(50/1.5)`）
- patch **128³**，`Resize` + `ScaleIntensityRangePercentiles(1, 99.9)`

**数据**：BIDS 布局下仅处理 **`sub-*/ses-*/anat/`** 与 **`sub-*/ses-*/dwi/`** 中的 **`.nii.gz`**（如 FLAIR、dwi、adc）；**跳过** `derivatives/`、`__MACOSX`。数据根默认为 `unzip/ISLES-2022/ISLES-2022/`（与当前解压结构一致）。

**输出**：所有 `.npy` 扁平放在与 `download/`、`unzip/` **平级**的 **`npy/`** 下；文件名形如 `anat_sub-strokecase0140_ses-0001_FLAIR_0_1.npy`（`anat`/`dwi` + BIDS 文件名 stem + 通道索引 + patch 序号）。

In [ ]:
# Cell 1: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: 安装依赖
!pip install -q nibabel monai

In [ ]:
# Cell 3: ISLES 预处理（与 TCIA / AMOS / proc_spatial_sequence 逻辑一致）
import os
import random
import numpy as np
import nibabel as nib
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

DRIVE_ISLES_UNZIP = '/content/drive/MyDrive/dataset/pretrain/ISLES/unzip'
ISLES_DATA_ROOT = os.path.join(DRIVE_ISLES_UNZIP, 'ISLES-2022', 'ISLES-2022')
DRIVE_SAVE_ROOT = '/content/drive/MyDrive/dataset/pretrain/ISLES/npy'

PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0

ALLOWED_MODALITY_DIRS = frozenset({'anat', 'dwi'})


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode='trilinear'),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume_isles(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size):
    image, affine = load_nii_data(image_file)
    path_parts = image_file.replace('\\', '/').split('/')
    ds_name = path_parts[-2] if len(path_parts) >= 2 else 'ISLES'
    nii_id = path_parts[-1].split('.nii.gz')[0].split('.nii')[0].split('.mha')[0]

    if len(image.shape) == 4:
        return []
    images = [image]
    n, patch_path_list = 0, []

    for image_index, image in enumerate(images):
        image = image.transpose((2, 1, 0))
        _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
        for i in range(_patch_num):
            n += 1
            patch_size = random.choice(patch_size_list)
            image_patch, _ = cut_patch(image, patch_size)
            image_patch = image_patch.transpose((2, 1, 0))
            image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
            save_name = os.path.join(save_root, '%s_%s_%s_%d.nii.gz' % (ds_name, nii_id, image_index, start_index + n))
            patch_path_list.append(save_name)
            np.save(save_name.replace('.nii.gz', '.npy'), image_patch)
    return patch_path_list


def collect_isles_nii_files(data_root):
    """仅 anat/ 与 dwi/ 下的 .nii.gz；排除 derivatives 与 __MACOSX。"""
    out = []
    if not os.path.isdir(data_root):
        return out
    for dirpath, _, files in os.walk(data_root):
        norm = dirpath.replace('\\', '/')
        if '__MACOSX' in norm:
            continue
        if 'derivatives' in norm.split('/'):
            continue
        if os.path.basename(dirpath) not in ALLOWED_MODALITY_DIRS:
            continue
        for f in files:
            if f.endswith('.nii.gz') or f.endswith('.nii'):
                out.append(os.path.join(dirpath, f))
    return sorted(out)


os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)
data_list = collect_isles_nii_files(ISLES_DATA_ROOT)
print(f'ISLES 数据根: {ISLES_DATA_ROOT}')
print(f'找到 {len(data_list)} 个 .nii/.nii.gz（仅 anat + dwi）')

if len(data_list) == 0:
    print('未找到文件，请检查 ISLES_DATA_ROOT 是否与解压结构一致（ISLES-2022/ISLES-2022/）。')
else:
    patch_list_all = []
    for i, path in enumerate(tqdm(data_list, desc='预处理')):
        patch_list = cut_and_save_one_volume_isles(path, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE)
        patch_list_all += patch_list
        if (i + 1) % 50 == 0:
            print(f'[{i+1}/{len(data_list)}] {path} -> {len(patch_list)} npy')
    print(f'\n完成! 共 {len(patch_list_all)} 个 npy, 输出: {DRIVE_SAVE_ROOT}')

In [ ]:
# Cell 4 (可选): 生成 train_patch_spatial.txt 用于 pretrain
list_save_dir = os.path.join(os.path.dirname(DRIVE_SAVE_ROOT), 'pretrain_lists')
os.makedirs(list_save_dir, exist_ok=True)
npy_files = sorted([os.path.join(DRIVE_SAVE_ROOT, f) for f in os.listdir(DRIVE_SAVE_ROOT) if f.endswith('.npy')])
with open(os.path.join(list_save_dir, 'train_patch_spatial.txt'), 'w') as f:
    f.write('\n'.join(npy_files))
print(f'train_patch_spatial.txt 已保存: {list_save_dir}')

In [4]:
# ISLES 预处理完成度校验（与 colab_isles_preprocess 一致）
import os
import nibabel as nib
from tqdm import tqdm

ISLES_DATA_ROOT = "/content/drive/MyDrive/dataset/pretrain/ISLES/unzip/ISLES-2022/ISLES-2022"
DRIVE_NPY_ROOT = "/content/drive/MyDrive/dataset/pretrain/ISLES/npy"
START_INDEX = 0
PATCH_NUM = 50
PATCH_Z_THR = 128

ALLOWED_MODALITY_DIRS = frozenset({"anat", "dwi"})


def collect_isles_nii_files(data_root):
    out = []
    if not os.path.isdir(data_root):
        return out
    for dirpath, _, files in os.walk(data_root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm:
            continue
        if "derivatives" in norm.split("/"):
            continue
        if os.path.basename(dirpath) not in ALLOWED_MODALITY_DIRS:
            continue
        for f in files:
            if f.endswith(".nii.gz") or f.endswith(".nii"):
                out.append(os.path.join(dirpath, f))
    return sorted(out)


def meta_from_nii_path(p):
    parts = p.replace("\\", "/").split("/")
    ds_name = parts[-2]
    nii_id = parts[-1].split(".nii.gz")[0].split(".nii")[0]
    return ds_name, nii_id


def z_len_after_transpose(nii_path):
    img = nib.load(nii_path).get_fdata()
    if len(img.shape) == 4:
        return None
    return img.transpose((2, 1, 0)).shape[0]


def expected_patches(z0):
    if z0 is None:
        return 0
    return int(PATCH_NUM / 1.5) if z0 < PATCH_Z_THR else PATCH_NUM


def parse_npy_stem(stem):
    # ..._{image_index}_{n}  —— 右侧两段固定为 0 与整数
    prefix, img_s, n_s = stem.rsplit("_", 2)
    if prefix.startswith("anat_"):
        ds_name, nii_id = "anat", prefix[len("anat_") :]
    elif prefix.startswith("dwi_"):
        ds_name, nii_id = "dwi", prefix[len("dwi_") :]
    else:
        raise ValueError("prefix 必须以 anat_ 或 dwi_ 开头")
    return ds_name, nii_id, int(img_s), int(n_s)


# ---------- 应有 patch 数（逐体积读 nii，较慢） ----------
nii_paths = collect_isles_nii_files(ISLES_DATA_ROOT)
expected = {}
for p in tqdm(nii_paths, desc="读 nii 推导应有 patch 数"):
    ds, nid = meta_from_nii_path(p)
    z0 = z_len_after_transpose(p)
    exp = expected_patches(z0)
    expected[(ds, nid)] = {"path": p, "z_after_T": z0, "n_patches": exp, "skipped_4d": z0 is None}

# ---------- 扫描 npy ----------
actual_counts = {k: set() for k in expected}
orphan_files = []
if not os.path.isdir(DRIVE_NPY_ROOT):
    print(f"缺少目录: {DRIVE_NPY_ROOT}")
else:
    for fn in os.listdir(DRIVE_NPY_ROOT):
        if not fn.endswith(".npy"):
            continue
        stem = fn[:-4]
        try:
            ds, nid, img_idx, n = parse_npy_stem(stem)
        except Exception:
            orphan_files.append(fn)
            continue
        key = (ds, nid)
        if key not in actual_counts:
            orphan_files.append(fn)
            continue
        if img_idx != 0:
            orphan_files.append(fn)
            continue
        lo = START_INDEX + 1
        hi = START_INDEX + expected[key]["n_patches"]
        if not (lo <= n <= hi):
            orphan_files.append(fn)
            continue
        actual_counts[key].add(n)

# ---------- 汇总 ----------
bad = []
for key, info in expected.items():
    if info["skipped_4d"]:
        bad.append((key, "4D 已跳过，不应有 patch", info))
        continue
    exp_n = info["n_patches"]
    need = set(range(START_INDEX + 1, START_INDEX + exp_n + 1))
    got = actual_counts[key]
    if got != need:
        bad.append(
            (
                key,
                f"应有 {exp_n} 个索引 {START_INDEX+1}..{START_INDEX+exp_n}，实际 {len(got)} 个",
                info,
            )
        )

print("=" * 72)
print(f"ISLES 数据根: {ISLES_DATA_ROOT}")
print(f"应处理的 nii 数量: {len(expected)}")
print(
    f"npy 目录下 .npy 数量: {len([f for f in os.listdir(DRIVE_NPY_ROOT) if f.endswith('.npy')]) if os.path.isdir(DRIVE_NPY_ROOT) else 0}"
)
print(f"无法匹配或索引异常的 .npy: {len(orphan_files)}")
if orphan_files[:20]:
    print("  样例:", orphan_files[:20])
print(f"体积 patch 不齐或与预期不符: {len(bad)}")
if bad[:15]:
    for key, msg, info in bad[:15]:
        print(f"  - {key}: {msg}")
        print(f"    {info['path']}")
    if len(bad) > 15:
        print(f"  ... 另有 {len(bad) - 15} 条")
print("=" * 72)

if not bad and not orphan_files:
    print("结论: 与当前预处理逻辑对照，ISLES 已全部对齐。")
else:
    print("结论: 存在问题（未跑完、改过 START_INDEX/PATCH_NUM、或 npy 命名不一致）。")

# 可选：列表文件
list_path = "/content/drive/MyDrive/dataset/pretrain/ISLES/pretrain_lists/train_patch_spatial.txt"
if os.path.isfile(list_path):
    with open(list_path, encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    npy_n = len([f for f in os.listdir(DRIVE_NPY_ROOT) if f.endswith(".npy")]) if os.path.isdir(DRIVE_NPY_ROOT) else 0
    ok = sum(1 for ln in lines if os.path.isfile(ln))
    print(f"\n列表: {list_path}")
    print(f"  非空行: {len(lines)} | 磁盘 .npy: {npy_n} | 列表中文件存在: {ok}")
else:
    print(f"\n未找到列表文件（未跑第 4 格可忽略）: {list_path}")

读 nii 推导应有 patch 数: 100%|██████████| 750/750 [09:40<00:00,  1.29it/s]


ISLES 数据根: /content/drive/MyDrive/dataset/pretrain/ISLES/unzip/ISLES-2022/ISLES-2022
应处理的 nii 数量: 750
npy 目录下 .npy 数量: 27334
无法匹配或索引异常的 .npy: 0
体积 patch 不齐或与预期不符: 0
结论: 与当前预处理逻辑对照，ISLES 已全部对齐。

未找到列表文件（未跑第 4 格可忽略）: /content/drive/MyDrive/dataset/pretrain/ISLES/pretrain_lists/train_patch_spatial.txt
